<a href="https://colab.research.google.com/github/leman-cap13/NLP_projects/blob/main/NaturalLanguageProgramming.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Sentiment Analysis Using Hugging Face Libraries

In [ ]:
from datasets import load_dataset


In [ ]:
imdb_dataset = load_dataset("stanfordnlp/imdb")

In [ ]:
imdb_dataset

In [ ]:
split = imdb_dataset["train"].train_test_split(train_size=0.8, seed=42)

imdb_train_set = split["train"]
imdb_valid_set = split["test"]
imdb_test_set = imdb_dataset["test"]

In [ ]:
print(imdb_train_set[1])

In [ ]:
print(imdb_train_set[1]["text"])


In [ ]:
print(imdb_train_set[1]["label"])

#Byte pair encoding (BPE)

In [ ]:
import tokenizers


In [ ]:
bpe_model = tokenizers.models.BPE(unk_token="<unk>")
bpe_model

In [ ]:
bpe_tokenizer = tokenizers.Tokenizer(bpe_model)
bpe_tokenizer

In [ ]:
bpe_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()


In [ ]:
special_tokens = ["<pad>", "<unk>"] # pad=0 , unk=1


In [ ]:
bpe_trainer = tokenizers.trainers.BpeTrainer(
    vocab_size=1000,
    special_tokens=special_tokens
)

In [ ]:
train_reviews = [review["text"].lower() for review in imdb_train_set]

train_reviews[0]

In [ ]:
bpe_tokenizer.train_from_iterator(train_reviews, bpe_trainer)

In [ ]:
some_review = "what an awesome movie! 😊"

bpe_encoding = bpe_tokenizer.encode(some_review)

In [ ]:
print(bpe_encoding.tokens)

In [ ]:
print(bpe_encoding.ids)

In [ ]:
print(bpe_encoding.type_ids)

In [ ]:
# w h a t
# 0 1 2 3

In [ ]:
print(bpe_encoding.offsets)

In [ ]:
print(bpe_encoding.attention_mask) # 0->padding 1 -> real token

In [ ]:
print(bpe_encoding.special_tokens_mask)

In [ ]:
print(bpe_tokenizer.decode(bpe_encoding.ids))

In [ ]:
bpe_encodings = bpe_tokenizer.encode_batch(train_reviews[:3])

In [ ]:
bpe_encodings[0].tokens

In [ ]:
bpe_tokenizer.enable_padding(
    pad_id=0,
    pad_token="<pad>"
)

bpe_tokenizer.enable_truncation(max_length=500)

In [ ]:
import torch

bpe_encodings = bpe_tokenizer.encode_batch(train_reviews[:3])

bpe_batch_ids = torch.tensor(
    [encoding.ids for encoding in bpe_encodings]
)

print(bpe_batch_ids)

In [ ]:
attention_mask = torch.tensor(
    [encoding.attention_mask for encoding in bpe_encodings]
)

print(attention_mask)

In [ ]:
lengths = attention_mask.sum(dim=-1)
print(lengths)

#Byte-level BPE / BBPE

In [ ]:
bpe_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.ByteLevel()


In [ ]:
bpe_tokenizer.decoder = tokenizers.decoders.ByteLevel()


In [ ]:
special_tokens = ["<pad>", "<unk>"]

bpe_trainer = tokenizers.trainers.BpeTrainer(
    vocab_size=1000,
    special_tokens=special_tokens
)

In [ ]:
bpe_tokenizer.train_from_iterator(
    train_reviews,
    bpe_trainer
)

In [ ]:
some_review = "what an awesome movie!"

encoding = bpe_tokenizer.encode(some_review)

In [ ]:
print(encoding.tokens) #Ġ -> soz baslangici
print(encoding.ids)
print(bpe_tokenizer.decode(encoding.ids))

#WordPiece

In [ ]:
wordpiece_model = tokenizers.models.WordPiece(
    unk_token="[UNK]"
)

wordpiece_tokenizer = tokenizers.Tokenizer(
    wordpiece_model
)

In [ ]:
wordpiece_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()


In [ ]:
wordpiece_trainer = tokenizers.trainers.WordPieceTrainer(
    vocab_size=1000,
    special_tokens=["[PAD]", "[UNK]"]
)

In [ ]:
wordpiece_tokenizer.train_from_iterator(
    train_reviews,
    wordpiece_trainer
)

In [ ]:
some_review = "what an awesome movie! 😊"

encoding = wordpiece_tokenizer.encode(some_review)

print("TOKENS:")
print(encoding.tokens)

print("\nIDS:")
print(encoding.ids)

print("\nDECODED:")
print(wordpiece_tokenizer.decode(encoding.ids))

In [ ]:
wordpiece_tokenizer.decoder = tokenizers.decoders.WordPiece(
    prefix="##"
)

wordpiece_trainer = tokenizers.trainers.WordPieceTrainer(
    vocab_size=1000,
    special_tokens=["[PAD]", "[UNK]"],
    continuing_subword_prefix="##"
)

wordpiece_tokenizer.train_from_iterator(
    train_reviews,
    wordpiece_trainer
)


some_review = "what an awesome movie! 😊"

encoding = wordpiece_tokenizer.encode(some_review)

print("TOKENS:")
print(encoding.tokens)

print("\nIDS:")
print(encoding.ids)

print("\nDECODED:")
print(wordpiece_tokenizer.decode(encoding.ids))

#Unigram LM

In [ ]:
unigram_model = tokenizers.models.Unigram()

unigram_tokenizer = tokenizers.Tokenizer(
    unigram_model
)


In [ ]:
unigram_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Metaspace(
    replacement="▁",
    prepend_scheme="always"
)


In [ ]:
unigram_tokenizer.decoder = tokenizers.decoders.Metaspace(
    replacement="▁",
    prepend_scheme="always"
)

In [ ]:
unigram_trainer = tokenizers.trainers.UnigramTrainer(
    vocab_size=1000,
    special_tokens=["<pad>", "<unk>"],
    unk_token="<unk>"
)


In [ ]:
unigram_tokenizer.train_from_iterator(
    train_reviews,
    unigram_trainer
)

In [ ]:
some_review = "what an awesome movie!😍"

encoding = unigram_tokenizer.encode(some_review)

print("TOKENS:")
print(encoding.tokens)

print("\nIDS:")
print(encoding.ids)

print("\nDECODED:")
print(unigram_tokenizer.decode(encoding.ids, skip_special_tokens=False))

#Reusing Pretrained Tokenizers

In [ ]:
import transformers

In [ ]:
gpt2_tokenizer=transformers.AutoTokenizer.from_pretrained("gpt2")
gpt2_tokenizer

In [ ]:
gpt2_encoding = gpt2_tokenizer(
    train_reviews[:3],
    truncation=True,
    max_length = 500)

In [ ]:
gpt2_encoding

In [ ]:
train_reviews[0]

In [ ]:
gpt2_token_ids = gpt2_encoding["input_ids"][0][:10]
print(gpt2_token_ids)

In [ ]:
gpt2_tokenizer.decode(gpt2_token_ids)

In [ ]:
bert_tokenizer = transformers.AutoTokenizer.from_pretrained("bert-base-uncased")
bert_tokenizer

In [ ]:
bert_encoding=bert_tokenizer(train_reviews[:3],
                             padding=True,
                             truncation=True,
                             max_length=500,
                             return_tensors="pt",
                            #  add_special_tokens=False
)

In [ ]:
bert_encoding

In [ ]:
bert_encoding["input_ids"] # starts with 101 [cls]  and ends with 102 [sep]

In [ ]:
bert_encoding["attention_mask"]

In [ ]:
albert_tokenizer = transformers.AutoTokenizer.from_pretrained("albert-base-v2")

In [ ]:
albert_encoding = albert_tokenizer(train_reviews[:3],
                                  padding=True,
                                  truncation=True,
                                  max_length=500)

In [ ]:
albert_encoding

In [ ]:
hf_tokenizer=transformers.PreTrainedTokenizerFast(
    tokenizer_object=bpe_tokenizer
)

In [ ]:
hf_encoding=hf_tokenizer(
    train_reviews[:3],
    padding=True,
    truncation=True,
    max_length=500
)

In [ ]:
hf_encoding

#Building and Training a Sentiment Analysis Model

In [ ]:
def collate_fn(batch, tokenizer=bert_tokenizer):
    reviews = [review["text"] for review in batch]
    labels = [review["label"] for review in batch]

    encodings = tokenizer(
        reviews,
        padding=True,
        truncation=True,
        max_length=200,
        return_tensors="pt"
    )

    labels = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)

    return encodings, labels

In [ ]:
from torch.utils.data import DataLoader


In [ ]:
batch_size=256
imdb_train_loader = DataLoader(
    imdb_train_set,
    batch_size=batch_size,
    collate_fn=collate_fn,
    shuffle=True
)

imdb_valid_loader = DataLoader(
    imdb_valid_set,
    batch_size=batch_size,
    collate_fn = collate_fn,
    shuffle=False

)

imdb_test_loader = DataLoader(
    imdb_test_set,
    batch_size=batch_size,
    collate_fn=collate_fn,
    shuffle=False
)

In [ ]:
import torch.nn as nn

In [ ]:
class SentimentAnalysisModel(nn.Module):
    def __init__(self, vocab_size, n_layers=2, embed_dim=2, hidden_dim=2, pad_id=0, dropout=0.2):
        super().__init__()

        self.embed= nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_id)

        self.gru = nn.GRU(
            embed_dim,
            hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout

        )
        self.output = nn.Linear(hidden_dim, 1)


    def forward( self, encodings):
        embeddings=self.embed(encodings["input_ids"])
        _outputs,hidden_states = self.gru(embeddings)
        return self.output(hidden_states[-1])


In [ ]:
import torch
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

In [ ]:
sequences = torch.tensor([
    [1, 2, 0, 0],
    [5, 6, 7, 8]
])

packed = pack_padded_sequence(
    sequences,
    lengths=(2, 4),
    enforce_sorted=False,
    batch_first=True
)

print(packed)

In [ ]:
padded, lengths = pad_packed_sequence(
    packed,
    batch_first=True
)

print(padded)
print(lengths)

In [ ]:
# import torch
# import torch.nn as nn
# from torch.nn.utils.rnn import pack_padded_sequence


class SentimentAnalysisModel(nn.Module):
    def __init__(
        self,
        vocab_size,
        n_layers=2,
        embed_dim=128,
        hidden_dim=64,
        pad_id=0,
        dropout=0.2
    ):
        super().__init__()

        self.embed = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_id
        )

        self.gru = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout
        )

        self.output = nn.Linear(hidden_dim, 1)

    def forward(self, encodings):
        input_ids = encodings["input_ids"]
        attention_mask = encodings["attention_mask"]

        embeddings = self.embed(input_ids)

        lengths = attention_mask.sum(dim=1)

        packed_embeddings = pack_padded_sequence(
            embeddings,
            lengths=lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        _outputs, hidden_states = self.gru(packed_embeddings)

        last_hidden_state = hidden_states[-1]

        logits = self.output(last_hidden_state)

        return logits

In [ ]:
vocab_size = bert_tokenizer.vocab_size
pad_id = bert_tokenizer.pad_token_id

model = SentimentAnalysisModel(
    vocab_size=vocab_size,
    n_layers=2,
    embed_dim=128,
    hidden_dim=64,
    pad_id=pad_id,
    dropout=0.2
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for encodings, labels in dataloader:
        encodings = {
            key: value.to(device)
            for key, value in encodings.items()
        }
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(encodings)

        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).float()

        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = correct / total

    return avg_loss, accuracy

In [ ]:
def evaluate(model, dataloader, criterion, device):
    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for encodings, labels in dataloader:
            encodings = {
                key: value.to(device)
                for key, value in encodings.items()
            }
            labels = labels.to(device)

            logits = model(encodings)

            loss = criterion(logits, labels)

            total_loss += loss.item()

            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).float()

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = correct / total

    return avg_loss, accuracy

In [ ]:
num_epochs = 5

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(
        model,
        imdb_train_loader,
        optimizer,
        criterion,
        device
    )

    valid_loss, valid_acc = evaluate(
        model,
        imdb_valid_loader,
        criterion,
        device
    )

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Valid Loss: {valid_loss:.4f} | "
        f"Valid Acc: {valid_acc:.4f}"
    )

In [ ]:
def predict_sentiment(model, tokenizer, sentences, device, max_length=200):
    model.eval()

    encodings = tokenizer(
        sentences,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )

    encodings = {
        key: value.to(device)
        for key, value in encodings.items()
    }

    with torch.no_grad():
        logits = model(encodings)
        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).long().squeeze(1)

    label_names = {
        0: "Negative",
        1: "Positive"
    }

    results = []

    for sentence, prob, pred in zip(sentences, probs.squeeze(1), preds):
        results.append({
            "sentence": sentence,
            "probability_positive": prob.item(),
            "predicted_label_id": pred.item(),
            "predicted_label": label_names[pred.item()]
        })

    return results

In [ ]:
test_sentences = [
    "This movie was amazing and I really enjoyed it.",
    "The film was boring and too long.",
    "I loved the acting but the story was weak.",
    "Absolutely terrible movie. I would not recommend it.",
    "One of the best films I have watched recently."
]

results = predict_sentiment(
    model=model,
    tokenizer=bert_tokenizer,
    sentences=test_sentences,
    device=device
)

for r in results:
    print("Sentence:", r["sentence"])
    print("Positive probability:", round(r["probability_positive"], 4))
    print("Predicted label id:", r["predicted_label_id"])
    print("Predicted label:", r["predicted_label"])
    print("-" * 50)

# Bidirectional RNNs

In [ ]:
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence


class SentimentAnalysisModel(nn.Module):
    def __init__(
        self,
        vocab_size,
        n_layers=2,
        embed_dim=128,
        hidden_dim=64,
        pad_id=0,
        dropout=0.2
    ):
        super().__init__()

        self.embed = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_id
        )

        self.gru = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout,
            bidirectional=True
        )

        self.output = nn.Linear(2 * hidden_dim, 1)

    def forward(self, encodings):
        input_ids = encodings["input_ids"]
        attention_mask = encodings["attention_mask"]

        embeddings = self.embed(input_ids)

        lengths = attention_mask.sum(dim=1)

        packed_embeddings = pack_padded_sequence(
            embeddings,
            lengths=lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        _outputs, hidden_states = self.gru(packed_embeddings)

        n_dims = self.output.in_features

        top_states = hidden_states[-2:].permute(1, 0, 2).reshape(-1, n_dims)

        logits = self.output(top_states)

        return logits

#Reusing Pretrained Embeddings and Language Models

In [ ]:
# first part only take first static layer of Bert and use gru model again

In [ ]:
bert_model=transformers.AutoModel.from_pretrained("bert-base-uncased")

In [ ]:
bert_model.embeddings.word_embeddings # vocab_size, embed_dim

In [ ]:
bert_model.embeddings.word_embeddings.weight.data

In [ ]:
nn.Embedding.from_pretrained(
    bert_model.embeddings.word_embeddings.weight.data,
    freeze=True
)

In [ ]:
class SentimentAnalysisModelPreEmbeds(nn.Module):
    def __init__(self, pretrained_embeddings, n_layers=2, hidden_dim=64, dropout=0.2 ):
        super().__init__()

        weights = pretrained_embeddings.weight.data

        self.embed = nn.Embedding.from_pretrained(
            weights,
            freeze=True
        )

        embed_dim = weights.shape[-1]

        self.gru = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout
        )

        self.output = nn.Linear(hidden_dim, 1)

    def forward(self, encodings):
        embeddings = self.embed(encodings["input_ids"])

        lengths = encodings["attention_mask"].sum(dim=1)

        packed = pack_padded_sequence(
            embeddings,
            lengths=lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        _outputs, hidden_states = self.gru(packed)

        return self.output(hidden_states[-1])

In [ ]:
imdb_model_bert_embeds = SentimentAnalysisModelPreEmbeds(
    bert_model.embeddings.word_embeddings
).to(device)
#.....train

In [ ]:
# use contextual bert model

In [ ]:
bert_encoding = bert_tokenizer(
    train_reviews[:3],
    padding=True,
    max_length=200,
    truncation=True,
    return_tensors="pt"
)
bert_model=transformers.AutoModel.from_pretrained("bert-base-uncased")

bert_output = bert_model(**bert_encoding)

In [ ]:
transformers.AutoModel.from_pretrained("bert-base-uncased").config.hidden_size

In [ ]:
class SentimentAnalysisModelBert(nn.Module):
    def __init__(self, n_layers=2, hidden_dim=64, dropout=0.2):
        super().__init__()

        self.bert = transformers.AutoModel.from_pretrained("bert-base-uncased")

        embed_dim = self.bert.config.hidden_size

        self.gru = nn.GRU(
            embed_dim,
            hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout
        )

        self.output = nn.Linear(hidden_dim, 1)

    def forward(self, encodings):
        contextualized_embeddings = self.bert(**encodings).last_hidden_state

        lengths = encodings["attention_mask"].sum(dim=1)

        packed = pack_padded_sequence(
            contextualized_embeddings,
            lengths=lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        _outputs, hidden_states = self.gru(packed) # output each step hidden state [1,2,3,x]

        return self.output(hidden_states[-1])

In [ ]:
#cls token

In [ ]:
class SentimentAnalysisModelBertCLS(nn.Module): # [CLS,1,2,3,4,5, SEP]. =<CLS represent sentence unil SEP
    def __init__(self):
        super().__init__()
        self.bert = transformers.AutoModel.from_pretrained("bert-base-uncased")
        self.output = nn.Linear(self.bert.config.hidden_size, 1)

    def forward(self, encodings):
        bert_output = self.bert(**encodings)
        return self.output(bert_output.last_hidden_state[:, 0, :]) # batch_size, seq_len, embed_dim seq_len=>token size

In [ ]:
# tokenizer = transformers.AutoTokenizer.from_pretrained("bert-base-uncased")

# model = SentimentAnalysisModelBertCLS()
# model.eval()

# text = "This movie was amazing!"

# encodings = tokenizer(
#     text,
#     return_tensors="pt",
#     padding=True,
#     truncation=True,
#     max_length=128
# )

# with torch.no_grad():
#     logit = model(encodings)
#     prob = torch.sigmoid(logit)

# print("logit:", logit.item())
# print("probability:", prob.item())
# print("prediction:", "Positive" if prob.item() >= 0.5 else "Negative")

In [ ]:
#pooler layer

In [ ]:
class SentimentAnalysisModelBertCLS(nn.Module):
    def __init__(self): # tanh(linear(cls)) for next sentence prediction [-1,1]
        super().__init__()
        self.bert = transformers.AutoModel.from_pretrained("bert-base-uncased")
        self.output = nn.Linear(self.bert.config.hidden_size, 1)

    def forward(self, encodings):
        bert_output = self.bert(**encodings)
        cls_vector = bert_output.pooler_output
        return self.output(cls_vector)

In [ ]:
# tokenizer = transformers.AutoTokenizer.from_pretrained("bert-base-uncased")

# model = SentimentAnalysisModelBertCLS().to(device)
# model.eval()

In [ ]:
# text = "This movie was very good. I loved it!"

# encodings = tokenizer(
#     text,
#     return_tensors="pt",
#     padding=True,
#     truncation=True,
#     max_length=128
# )

# encodings = {key: value.to(device) for key, value in encodings.items()}


In [ ]:
# with torch.no_grad():
#     logit = model(encodings)
#     prob = torch.sigmoid(logit)
#     label = 1 if prob.item() >= 0.5 else 0

# print("Logit:", logit.item())
# print("Probability:", prob.item())
# print("Label:", label)

# if label == 1:
#     print("Prediction: Positive")
# else:
#     print("Prediction: Negative")

#Task-Specific Classes

In [ ]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")


In [ ]:
bert_for_binary_clf = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2,
    dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)

In [ ]:
review = ["This was not a great movie!"]


encoding = bert_tokenizer(
    review,
    padding=True,
    truncation=True,
    max_length=200,
    return_tensors="pt"
)

In [ ]:
encoding = {key: value.to(device) for key, value in encoding.items()}

with torch.no_grad():
    output = bert_for_binary_clf(**encoding)

logits = output.logits
probs = torch.softmax(logits, dim=-1)
predicted_class = torch.argmax(probs, dim=-1).item()

In [ ]:
label_names = ["negative", "positive"]


In [ ]:
logits

In [ ]:
probs

In [ ]:
label_names[predicted_class]

#The Trainer API

In [ ]:
import torch
import numpy as np

In [ ]:
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")


In [ ]:
bert_for_binary_clf = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
).to(device)

In [ ]:
def tokenize_batch(batch):
    return bert_tokenizer(
        batch["text"],
        truncation=True,
        max_length=200
    )

In [ ]:
tok_imdb_train_set = imdb_train_set.map(tokenize_batch, batched=True)
tok_imdb_valid_set = imdb_valid_set.map(tokenize_batch, batched=True)
tok_imdb_test_set = imdb_test_set.map(tokenize_batch, batched=True)

In [ ]:
def compute_accuracy(pred):
    logits = pred.predictions
    labels = pred.label_ids

    predictions = np.argmax(logits, axis=-1)
    accuracy = (predictions == labels).mean()

    return {"accuracy": accuracy}

In [ ]:
!pip install wandb

In [ ]:
import wandb
wandb.login()



In [ ]:
train_args = TrainingArguments(
    output_dir="my_imdb_model", # it holds checkpoints and logs...
    num_train_epochs=2,  # pass 2 times for whole dataset
    per_device_train_batch_size=16, # for each step model takes 16 reviews
    per_device_eval_batch_size=16,
    eval_strategy="epoch", # her epoch sonuna validation acc yazilir
    logging_strategy="epoch",
    save_strategy="epoch",  #checkpoint
    load_best_model_at_end=True, # not last one but the best one
    metric_for_best_model="accuracy",
    report_to="wandb",
    run_name="bert-imdb-run-1"
)

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=bert_tokenizer)


In [ ]:
trainer = Trainer(
    model=bert_for_binary_clf,
    args=train_args,
    train_dataset=tok_imdb_train_set,
    eval_dataset=tok_imdb_valid_set,
    compute_metrics=compute_accuracy,
    data_collator=data_collator
)

In [ ]:
train_output = trainer.train()

In [ ]:
trainer.evaluate(tok_imdb_valid_set)

In [ ]:
review = ["This was not a great movie!"]

inputs = bert_tokenizer(
    review,
    return_tensors="pt",
    truncation=True,
    max_length=200,
    padding=True
)

inputs = {k: v.to(device) for k, v in inputs.items()}

bert_for_binary_clf.eval()

with torch.no_grad():
    outputs = bert_for_binary_clf(**inputs)

probs = torch.softmax(outputs.logits, dim=-1)
pred = torch.argmax(probs, dim=-1).item()

label_names = ["negative", "positive"]

print("Probabilities:", probs)
print("Prediction:", label_names[pred])

#Hugging Face Pipelines

In [ ]:
from transformers import pipeline

model_name = "distilbert-base-uncased-finetuned-sst-2-english"

classifier = pipeline(
    task="sentiment-analysis",
    model=model_name,
    truncation=True,
    max_length=512
)

texts = [
    "This was a great movie!",
    "This was not a great movie!",
    "The acting was amazing but the story was boring."
]

results = classifier(texts)

print(results)

In [ ]:
from transformers import pipeline

qa = pipeline("question-answering")

context = """
BERT is a transformer-based model developed for natural language understanding.
It uses token embeddings, positional embeddings, and attention mechanisms.
"""

question = "What does BERT use?"

result = qa(
    question=question,
    context=context
)

print(result)

In [ ]:
from transformers import pipeline

ner = pipeline(
    "ner",
    aggregation_strategy="simple"
)

text = "Laman works as an AI Engineer in Baku and studies NLP with Hugging Face."

result = ner(text)

print(result)

In [ ]:
import transformers
print(transformers.__version__)

In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="gpt2"
)

prompt = "In the future, artificial intelligence will"

result = generator(
    prompt,
    max_new_tokens=50,
    num_return_sequences=1
)

print(result[0]["generated_text"])

In [ ]:
from transformers import pipeline

fill_mask = pipeline(
    "fill-mask",
    model="bert-base-uncased"
)

text = "The capital of France is [MASK]."

result = fill_mask(text)

for item in result:
    print(item["sequence"], item["score"])

In [ ]:
from transformers import pipeline

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

texts = [
    "The GPU memory is not enough for training BERT.",
    "The football match ended with a dramatic goal.",
    "The recipe requires flour, eggs, and milk."
]

labels = ["machine learning", "sports", "food"]

for text in texts:
    result = classifier(text, candidate_labels=labels)
    print(text)
    print(result["labels"][0], result["scores"][0])
    print("-" * 80)